# Create a journal analysis project

Run the cells in order, then use the form below. Search for any journal by title or ISSN and **select the intended journal**. Choose an inclusive date range and the number of random papers, then click **Create project**.

The project contains the full metadata export, a random sample and the text-cleaning notebook. Choose your own folders and files for PDFs and analysis results. Existing projects are never overwritten.

Install dependencies in a terminal with `python3 -m pip install -r requirements.txt`, or uncomment the setup command in the next cell. Keep this notebook alongside `journal_tools.py` and `clean_and_count_text.ipynb`.

In [ ]:
# Uncomment if notebook dependencies are missing, then restart the kernel:
# %pip install -r requirements.txt

In [ ]:
from datetime import datetime
from html import escape
from pathlib import Path
from zoneinfo import ZoneInfo

import ipywidgets as widgets
from IPython.display import display
from app import journal_tools as tools

TOOLS_ROOT = Path(tools.__file__).resolve().parent.parent
TODAY = datetime.now(ZoneInfo("Europe/Zurich")).date()
DEFAULT_PROJECTS_ROOT = TOOLS_ROOT.parent


In [ ]:
field_style = {"description_width": "150px"}
wide = widgets.Layout(width="100%")
journal_query = widgets.Text(description="Journal title / ISSN:", placeholder="Enter a journal title or ISSN", layout=wide, style=field_style)
email_field = widgets.Text(description="Contact email:", placeholder="Optional, used only for Crossref requests", layout=wide, style=field_style)
search_button = widgets.Button(description="Find journal", button_style="info")
journal_choice = widgets.Dropdown(description="Select journal:", options=[], value=None, layout=wide, style=field_style)
start_field = widgets.DatePicker(description="Start date:", value=tools.months_before(TODAY, 12), style=field_style)
end_field = widgets.DatePicker(description="End date:", value=TODAY, style=field_style)
count_field = widgets.BoundedIntText(description="Random papers:", value=10, min=1, max=1000000, style=field_style)
basis_field = widgets.Dropdown(description="Publication date:", options=[("First known publication", "first"), ("Online publication", "online"), ("Print publication", "print")], value="first", style=field_style, layout=wide)
root_field = widgets.Text(description="Projects parent:", value=str(DEFAULT_PROJECTS_ROOT), style=field_style, layout=wide)
folder_field = widgets.Text(description="Project folder:", placeholder="Optional: leave empty for an automatic date + journal + range name", style=field_style, layout=wide)
seed_field = widgets.Text(description="Random seed:", placeholder="Optional integer; empty = a fresh recorded seed", style=field_style, layout=wide)
notices_field = widgets.Checkbox(description="Include notices and editorials in the sample", value=False, indent=False)
archive_field = widgets.Text(description="Archived JSON:", placeholder="Optional path to a saved *_crossref.json for an offline replay", style=field_style, layout=wide)
create_button = widgets.Button(description="Create project", button_style="primary", disabled=True, layout=widgets.Layout(width="180px"))
preview = widgets.HTML()
search_output = widgets.Output()
creation_output = widgets.Output()
last_project = None


def update_preview(change=None):
    journal = journal_choice.value
    create_button.disabled = journal is None
    if journal is None:
        preview.value = "<p>Find and select a journal to preview the project folder.</p>"
        return
    try:
        tools.validate_window(start_field.value, end_field.value, basis_field.value)
        name = folder_field.value.strip() or tools.project_folder_name(journal, start_field.value, end_field.value)
        path = Path(root_field.value).expanduser() / name
        preview.value = f"<p><strong>New project:</strong> {escape(str(path))}</p>"
    except ValueError as error:
        preview.value = f"<p>{escape(str(error))}</p>"
        create_button.disabled = True


def search_journal(button=None):
    search_button.disabled = True
    journal_choice.options = []
    search_output.clear_output()
    try:
        with search_output:
            print("Looking up journals…")
            candidates = tools.find_journals(journal_query.value, email_field.value.strip() or None)
            if not candidates:
                print("No journals found. Try a more specific title or an ISSN.")
            else:
                journal_choice.options = [("Choose the intended journal…", None)] + [
                    (f"{journal.name} ({', '.join(journal.issns)})", journal) for journal in candidates
                ]
                print(f"Found {len(candidates)} candidate(s). Select the intended journal below.")
    except Exception as error:
        with search_output:
            print(f"Journal lookup failed: {error}")
    finally:
        search_button.disabled = False
        update_preview()


def query_changed(change):
    # A changed search must not reuse a journal selected for the previous query.
    journal_choice.options = []
    search_output.clear_output()
    update_preview()


def make_project(button=None):
    global last_project
    create_button.disabled = True
    search_button.disabled = True
    creation_output.clear_output()
    try:
        with creation_output:
            if journal_choice.value is None:
                raise ValueError("Find and select a journal first.")
            if not root_field.value.strip():
                raise ValueError("Enter the parent folder for your projects.")
            seed = int(seed_field.value.strip()) if seed_field.value.strip() else None
            print("Fetching metadata and preparing the project. This can take a few minutes…")
            result = tools.create_project(
                journal_choice.value, start_field.value, end_field.value,
                count_field.value, root_field.value.strip(),
                date_basis=basis_field.value, seed=seed,
                include_notices=notices_field.value,
                email=email_field.value.strip() or None,
                project_name=folder_field.value.strip() or None,
                input_json=archive_field.value.strip() or None,
            )
            last_project = result
            print(f"Created: {result['path']}")
            print(f"Full export: {len(result['rows'])} records; eligible: {result['selection_report']['eligible_entry_count']}")
            print(f"Selected papers: {len(result['selected'])}; saved seed: {result['config']['seed']}")
            for index, row in enumerate(result['selected'], 1):
                print(f"{index}. {row['title']} ({row['doi']})")
            print("Open the new project's clean_and_count_text.ipynb. Organize PDFs and analysis results as you prefer.")
    except Exception as error:
        with creation_output:
            print(f"Project was not created: {error}")
    finally:
        search_button.disabled = False
        update_preview()


search_button.on_click(search_journal)
create_button.on_click(make_project)
journal_query.observe(query_changed, names="value")
for field in (journal_choice, start_field, end_field, basis_field, root_field, folder_field):
    field.observe(update_preview, names="value")
update_preview()
advanced = widgets.Accordion(children=[widgets.VBox([basis_field, seed_field, notices_field, archive_field])], selected_index=None)
advanced.set_title(0, "Publication date, reproducibility and offline replay")
display(widgets.VBox([
    journal_query, email_field, search_button, search_output, journal_choice,
    widgets.HBox([start_field, end_field]), count_field,
    root_field, folder_field, advanced, preview, create_button, creation_output,
]))

## Generated structure

```text
<creation-date>_journalanalysis_<journal>_<start>_<end>/
├── project.json
├── README.md
├── clean_and_count_text.ipynb
└── output/
    ├── <journal>_<start>_<end>.csv
    ├── <journal>_<start>_<end>.bib
    ├── <journal>_<start>_<end>_crossref.json
    ├── <journal>_<start>_<end>_report.json
    ├── <journal>_<start>_<end>_random<N>.csv
    ├── <journal>_<start>_<end>_random<N>.bib
    └── <journal>_<start>_<end>_random<N>_selection.json
```

The default parent is the tools folder's parent, so new projects sit alongside the tools folder. You can change it in the form. Folder dates use Europe/Zurich. No PDFs are downloaded. Choose your own folders and files for PDFs and analysis results.

The default date basis is the **earliest known publication date** across online, print, published and issued metadata. Dates are inclusive. Partial dates use the first day for filtering while exports keep their original precision. CSV and BibTeX contain the same DOIs and citation keys in the same order.

The sample is uniform without replacement across eligible DOIs. Obvious notices are excluded using a title heuristic, and all exclusions and the actual seed are recorded. Review those exclusions and compare the text-cleaning results with the source PDF. Crossref coverage depends on publisher deposits. [Crossref REST API documentation](https://www.crossref.org/documentation/retrieve-metadata/rest-api/).

## Optional: create a project from code

The form uses the same function as the example below. For an offline replay, construct `tools.Journal("Journal title", ("1234-5679",))` with the real journal identifiers and pass a path as `input_json`. This also avoids a journal-lookup request. Keep the creation call commented until you have chosen your settings.

In [ ]:
# from datetime import date
# candidates = tools.find_journals("Your journal title or ISSN")
# for index, candidate in enumerate(candidates):
#     print(index, candidate)
# journal = candidates[0]  # Select the intended journal after reviewing the list.
# result = tools.create_project(
#     journal=journal,
#     start=date(2025, 11, 4),
#     end=date(2026, 11, 4),
#     count=10,
#     projects_root=DEFAULT_PROJECTS_ROOT,
#     seed=42,  # Optional: omit for a fresh recorded seed.
#     # input_json=Path("path/to/saved_crossref.json"),
# )

## Optional: draw another sample without fetching again

This writes a separate sample alongside the archived source and preserves the first sample. To create another complete analysis project, replay the archived JSON into a new project folder instead.

In [ ]:
# import csv
# import json
# if last_project is None:
#     raise ValueError("Create a project first, or set project_path to an existing project.")
# project_path = last_project["path"]
# config = json.loads((project_path / "project.json").read_text(encoding="utf-8"))
# source = project_path / config["full_export"]
# with source.open(encoding="utf-8-sig", newline="") as handle:
#     population = list(csv.DictReader(handle))
# selected, report = tools.sample_rows(population, count=10)  # Optional seed=42.
# stem = f"{source.stem}_random{len(selected)}_seed{report['seed']}"
# if any((source.parent / (stem + suffix)).exists() for suffix in (".csv", ".bib", "_selection.json")):
#     raise FileExistsError("This sample already exists; choose another seed.")
# tools.write_selection(selected, source.parent, stem, dict(report, source_csv=config["full_export"]))
# print(source.parent / (stem + ".csv"))